In [7]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import numpy as np
import plot_utils as pu
import timeit as ti

In [2]:
# configure the backend for matplotlib
# this one should allow zoom:
%matplotlib widget
# to make that work you need: "pip install ipympl" and run "jupyter nbextension enable --py widgetsnbextension"

# this will work without the above dependencies but won't allow zoom
# %matplotlib inline

# this option may work whenever they fix bugs in mpld3
# import mpld3
# mpld3.enable_notebook()

In [ ]:
# A function to create cuts given a target point
def add_balas_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx, plotter):
    # for each column in the tableau
    # construct a sparse vector for it
    # get the length of that vector via norm1 (plus 1 if we're an int column)
    # add our cut: sum_j(x_j/a_j)
    
    norm = 2
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    if len(current) < 7:
        print("   Gap to target:", radius, ", Score:", relaxed.getObjective().getValue(), ",", current, "to", target)
    else:
        print("   Gap to target:", radius, ", Score:", relaxed.getObjective().getValue())
    if plotter is not None:
        plotter.add_ball(radius)

    variables = relaxed.getVars()  # TODO: pass this in as it's expensive?
    constraints = relaxed.getConstrs()  # wish we didn't have to use this one
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=1)
    negated_vars = [basis[nr] for nr in negated_rows]
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)  # TODO: don't even bother to read them in
    basis = np.delete(basis, to_drop)

    # Conforti has negative vectors with 1 at row=col, with the rest negated.
    # However, empirically, it seems that the opposite is what we want (gurobi-specific or ineq. standardization issue)
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
    lengths = np.linalg.norm(tableau, norm, axis=0)
    lengths /= radius
    
    def find_variable(index):
        if index < len(variables):
            # handle inverted variables (SCIP and Gurobi both have this silliness)
            if variables[index].VBasis == -2:  # not yet sure this is correct
                return variables[index].X - variables[index]
            assert variables[index].VBasis == -1  # not handling -3 yet
            return variables[index]
        # if only gurobi gave us access to their slack variables...
        # instead, we have to solve for it:
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        lhs, sense, rhs = relaxed.getRow(constraint), constraint.Sense, constraint.RHS
        if sense == '<':
             return rhs - lhs
        elif sense == '>':
            return lhs - rhs
        else:
            return 0.0  # is this right for equality constraints?
    
    summed_terms = gp.quicksum(lengths[i] * find_variable(j) for i, j in enumerate(col_to_var))
    new_con = relaxed.addConstr(summed_terms >= 1)
    
    # what does it mean to take a relaxed point in full space and compare only its want-to-be-integer components to another?
    if plotter is not None:
        relaxed.update()
        plotter.add_constraint(new_con)
    return True

# a function to run cuts against the nearest integer:
def run_cuts_to_nearest_int(instances, cut_function, loops=10, graph_2D_3D=True):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.params.Method = 1
        plotter = pu.create(model) if graph_2D_3D else None
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        # gu.standardize_lt_to_gt(model)
        # cons1 = gu.standardize_ub_to_constr(model)
        # cons2 = gu.standardize_lb_to_constr(model)

        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break
            nearest = gu.nearest_integer(int_vars)
            if not cut_function(model, nearest, int_vars, int_idx, plotter):
                break
        if model.Status == gp.GRB.OPTIMAL:
            print("   Final score of relaxed:", model.getObjective().getValue())
        if plotter is not None:
            plotter.render()

# test the cuts on simple examples:
run_cuts_to_nearest_int(list(el.get_instances().values()), add_balas_ball_cut)

In [ ]:
miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76
run_cuts_to_nearest_int(miplib_subset, add_balas_ball_cut)

In [ ]:
import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz5']]
run_cuts_to_nearest_int(jsplib_subset, add_balas_ball_cut)
# jsplib_subset[0].as_gurobi_model().optimize()

In [8]:
# run comparisons:
def exit_early(model: gp.Model, where):
    if where == gp.GRB.Callback.MIPNODE:
        if model.cbGet(gp.GRB.Callback.MIPNODE_NODCNT) > 0:
            dualBound = model.cbGet(gp.GRB.Callback.MIPNODE_OBJBND)
            primalBound = model.cbGet(gp.GRB.Callback.MIPNODE_OBJBST)
            print("   Gurobi's best dual bound:", dualBound, ", best primal:", primalBound)
            model.terminate()

def compare_cuts(instances):
    for instance in instances:
        mi = instance.as_gurobi_model()
        if mi.NumVars * mi.NumConstrs * 8 > (1<<32):
            print("Skipping {mi.ModelName} because it is too big for us")
            continue
        
        print("Attempting Gurobi's aggressive cuts (10 passes):")
        mi.params.Cuts = 3
        mi.params.CutPasses = 10
        mi.params.LogToConsole = 0
        start = ti.default_timer()
        mi.optimize(exit_early)
        print(f"   Done checking root node in {ti.default_timer() - start}ms")

        print()
        print("Attempting Gurobi's NoRel (8 seconds' worth):")
        del mi
        mi = instance.as_gurobi_model()
        mi.params.Heuristics = 0
        mi.params.NoRelHeurTime = 8.0
        mi.params.Cuts = 0
        mi.params.CutPasses = 0
        mi.params.LogToConsole = 0
        mi.optimize(exit_early)
        
        print("Expected optimum:", instance.score)
        print()
        run_cuts_to_nearest_int([instance], add_balas_ball_cut, loops=8, graph_2D_3D=False)
        print()

import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz5']]

miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76

example_instances = el.get_instances()

compare_cuts(miplib_instances.values())

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Read MPS format model from file mip2017_benchmark\revised-submissions\miplib2010_50v-10\instances\50v-10.mps.gz
Reading time = 0.06 seconds
50v-10: 233 rows, 2013 columns, 2745 nonzeros
Running model: 50v-10
   Relaxed 1647 variables on 50v-10
   Gap to target: 1.8035720348929818 , Score: 2879.06568

MemoryError: Unable to allocate 1.35 TiB for an array with shape (295990, 624564) and data type float64